## Hypothesis
Extreme negative sentiment (bottom decile) is assc. w/ *statistically significant*
increases in downside risk characterized by:
- lower forward returns
- higher forward volatility



### Research Shift (Update)
Prior analysis indicates that sentiment does *not* provide reliable short-term predictive signal for returns when evaluated symmetrically.

As a result, this hypothesis shifts to sentiment as a **conditional risk indicator**

Sentiment is evaluated in conjunction w/ momentum, rather than as a standalone signal

`price_return ~ momentum + sentiment`

## Origin

Based on the results of extended lag and aggregation analysis:
- Extended lag and aggregation analysis suggests that sentiment does **not** provide reliable short-term predictive signal for returns.
- Extreme sentiment events exhibit asymmetric behavior. 
    - Negative sentiment retains statistical significance under permutation testing and aligns with downside movements, while positive sentiment appears largely explained by underlying market drift. 

## Question

Does extreme negative sentiment (bottom decile) lead to:
1. Lower forward returns relative to baseline?
2. Higher forward volatility?
3. Increased probability of negative return events?
4. Are these effects statistically significant under permutation testing?

## Assumptions

- **Data window**: 2024-02-23 - 2026-02-23
- **Aggregation**: Rolling sentiment (7d,14d) and raw daily sentiment
- **Filtering**: Sentiment segregated into deciles (bottom 10% focus)
- **Forward horizons:**: t+7, t+14, t+30 returns
- **Risk metrics**:
    - Mean forward return
    - Standard deviation (volatility)
    - Probability of negative return
- **Validation**: Permutation testing (2000 iterations) on extreme sentiment buckets


## Data Loading (minimal)

Load only the essential data needed for this analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from msa.utils.paths import get_joined_dataset, get_processed_data_path
from msa.utils.vars import WINDOW_START, WINDOW_END, MAG7_TICKERS

# Load data
INPUT_PATH = get_joined_dataset()
df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["date", "article_date", "price_date"]
)

print(f"Loaded {len(df):,} rows from {INPUT_PATH.name}")
df.head()

Loaded 89,405 rows from gdelt_ohlcv_join.csv


,seendate,url,title,language,domain,socialimage,company,ticker,date,sentiment_score,sentiment_confidence,sentiment_label,article_date,price_date,next_open,next_high,next_low,next_close,next_adj_close,next_volume
0,2024-02-08 00:00:00+00:00,https://www.businesstimes.com.sg/companies-mar...,Arm soars after expansion into new markets buo...,English,businesstimes.com.sg,https://static1.businesstimes.com.sg/s3fs-publ...,Apple,AAPL,2024-02-08,0.815164,0.877368,positive,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
1,2024-02-08 00:45:00+00:00,https://www.hucknalldispatch.co.uk/lifestyle/f...,Dutch Barn Orchard Vodka achieves nationwide l...,English,hucknalldispatch.co.uk,https://www.hucknalldispatch.co.uk/webimg/b25l...,Apple,AAPL,2024-02-08,0.927444,0.938555,positive,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
2,2024-02-08 00:45:00+00:00,https://www.fool.com/earnings/call-transcripts...,PayPal ( PYPL ) Q4 2023 Earnings Call Transcript,English,fool.com,NaN,Apple,AAPL,2024-02-08,-0.010950,0.934771,neutral,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
3,2024-02-08 01:15:00+00:00,https://invezz.com/news/2024/02/07/disney-q1-e...,Disney Q1 earnings : dividend increased as DTC...,English,invezz.com,https://invezz.com/wp-content/uploads/2022/11/...,Apple,AAPL,2024-02-08,0.838071,0.908904,positive,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200
4,2024-02-08 01:15:00+00:00,https://www.nbcnewyork.com/news/business/money...,"Jim Cramer says recent moves in Apple , Chipot...",English,nbcnewyork.com,https://media.nbcnewyork.com/2023/11/107113454...,Apple,AAPL,2024-02-08,-0.320946,0.543592,neutral,2024-02-08,2024-02-09,188.649994,189.990005,188.0,188.850006,187.146774,45155200


# Data Filtering & Preparation

Apply filters and prepare data with clear tracking of each step.

In [11]:
print("="*70)
print("DATA FILTERING PIPELINE")
print("="*70)

#  Initial dataset
print(f"\n1. Initial dataset: {len(df):,} rows")

# Restrict to project window (price_date = day we have returns)
df_filtered = df[
    (df["price_date"].dt.normalize() >= pd.Timestamp(WINDOW_START))
    & (df["price_date"].dt.normalize() <= pd.Timestamp(WINDOW_END))
].copy()
print(f"2. After date window ({WINDOW_START} to {WINDOW_END}): {len(df_filtered):,} rows")

# Filter to mag7
df_mag7 = df_filtered[df_filtered["ticker"].isin(MAG7_TICKERS)]
print(f"3. After mag7 filter: {len(df_mag7):,} rows")

# Filter to negative sentiment
df_mag7_neg = df_mag7[df_mag7["sentiment_label"] == "negative"]

print("\n" + "="*70)
df_filtered = df.copy()  # Placeholder - replace with your filtering logic

DATA FILTERING PIPELINE

1. Initial dataset: 89,405 rows
2. After date window (2024-02-23 to 2026-02-23): 88,851 rows
3. After mag7 filter: 88,851 rows



# Calculations

Core metrics and calculations for testing the hypothesis.

In [64]:
# --- 1) Daily close-to-close returns per (price_date, ticker) ---
# Article rows don't have return_1d; build one row per trading day from the join OHLC.
returns_daily = (
    df_mag7.groupby(["price_date", "ticker"], as_index=False)
    .agg(close=("next_close", "first"))
)
returns_daily["price_date"] = pd.to_datetime(returns_daily["price_date"]).dt.normalize()
returns_daily = returns_daily.sort_values(["ticker", "price_date"]).reset_index(drop=True)
returns_daily["return_1d"] = returns_daily.groupby("ticker")["close"].pct_change()

# Forward return = next trading day's return (shift on the DAILY panel, not on articles)
returns_daily["forward_return_1d"] = returns_daily.groupby("ticker")["return_1d"].shift(-1)

# Calculate forward return method
def _forward_cumulative_return(series: pd.Series, h: int) -> pd.Series:
    """For each row i, cumulative return from i+1 .. i+h (h trading days ahead)."""
    arr = series.to_numpy(dtype=float)
    n = len(arr)
    out = np.full(n, np.nan)
    for i in range(n):
        chunk = arr[i + 1 : i + 1 + h]
        if len(chunk) < h or np.any(np.isnan(chunk)):
            continue
        out[i] = float(np.prod(1.0 + chunk) - 1.0)
    return pd.Series(out, index=series.index)

# Yields forward returns for 7d, 14d, 30d
for h, colname in [(7, "forward_return_7d"), (14, "forward_return_14d"), (30, "forward_return_30d")]:
    returns_daily[colname] = returns_daily.groupby("ticker", group_keys=False)["return_1d"].transform(
        lambda s, hh=h: _forward_cumulative_return(s, hh)
    )

# --- 2) Attach returns to negative-sentiment article rows ---
neg = df_mag7_neg.copy()
neg["price_date"] = pd.to_datetime(neg["price_date"]).dt.normalize()

merge_cols = [
    "price_date",
    "ticker",
    "return_1d",
    "forward_return_1d",
    "forward_return_7d",
    "forward_return_14d",
    "forward_return_30d",
]
neg_with_ret = neg.merge(returns_daily[merge_cols], on=["price_date", "ticker"], how="left")

# --- 3) Bottom decile of negative sentiment (lowest 10% of sentiment_score among negative articles) ---
neg_sorted = neg_with_ret.sort_values("sentiment_score", ascending=True)
cutoff = neg_sorted["sentiment_score"].quantile(0.1)
df_mag7_neg_decile = neg_sorted[neg_sorted["sentiment_score"] <= cutoff].copy()

print(f"Negative articles: {len(neg_with_ret):,}; bottom decile (<= {cutoff:.4f}): {len(df_mag7_neg_decile):,}")
print(df_mag7_neg_decile[["forward_return_1d", "forward_return_7d", "forward_return_14d", "forward_return_30d"]].describe())


Negative articles: 11,145; bottom decile (<= -0.9445): 1,115
       forward_return_1d  forward_return_7d  forward_return_14d  \
count        1114.000000        1046.000000          926.000000   
mean            0.006014           0.043660            0.102698   
std             0.046430           0.119208            0.179473   
min            -0.177012          -0.208467           -0.323819   
25%            -0.017311          -0.044878           -0.022495   
50%             0.003925           0.027249            0.074238   
75%             0.021717           0.101960            0.220149   
max             0.241050           0.417604            0.671801   

       forward_return_30d  
count          755.000000  
mean             0.187905  
std              0.261881  
min             -0.317338  
25%              0.007618  
50%              0.145689  
75%              0.270920  
max              1.041797  


# Test

Evaluate the hypothesis through statistical tests and visualizations.

### Conditional Risk Test (Core)

Evaluate whether extreme negative sentiment increases:

1. Probability of next-day downside
2. Magnitude of negative returns
3. Statistical significance of difference vs baseline

In [55]:
# Calculate the bottom decile threshold
low_threshold = df_mag7_neg_decile["sentiment_score"].quantile(0.1)
high_threshold = df_mag7_neg_decile["sentiment_score"].quantile(0.9)

# Create a binary indicator for sentiment in the (bottom decile)
df_mag7_neg_decile["neg_extreme"] = df_mag7_neg_decile["sentiment_score"] <= low_threshold
# Calculate the next day's return
df_mag7_neg_decile['next_return'] = df_mag7_neg_decile.groupby('ticker')['return_1d'].shift(-1)
# Create a binary indicator for downside returns
df_mag7_neg_decile['downside'] = df_mag7_neg_decile['next_return'] < 0
# Create a binary indicator for high sentiment (top decile)
df_mag7_neg_decile['neg_high'] = df_mag7_neg_decile['sentiment_score'] >= high_threshold

# Calculate the mean downside probability for each sentiment group
p_down_prob = df_mag7_neg_decile[df_mag7_neg_decile['neg_extreme']]['downside'].mean()
p_down_all = df_mag7_neg_decile['downside'].mean()
p_down_prob_high = df_mag7_neg_decile[df_mag7_neg_decile['neg_high']]['downside'].mean()

# Print results
print(f"Probability of negative return: {p_down_all:.4f}")
print(f"Probability of negative return (bottom decile): {p_down_prob:.4f}")
print(f"Probability of negative return (top decile): {p_down_prob_high:.4f}")
print("="*70)

# Perform t-test
from scipy.stats import ttest_ind

neg = df_mag7_neg_decile[df_mag7_neg_decile['neg_extreme']]['next_return'].dropna()
not_neg = df_mag7_neg_decile[~df_mag7_neg_decile['neg_extreme']]['next_return'].dropna()
base = df_mag7_neg_decile['next_return'].dropna()
pos = df_mag7_neg_decile[df_mag7_neg_decile['neg_high']]['next_return'].dropna()

# Calculate means for each group
neg_mean = neg.mean()
not_neg_mean = not_neg.mean()
base_mean = base.mean()
pos_mean = pos.mean()

# Calculate volatility for each group
neg_vol = neg.std()
not_neg_vol = not_neg.std()
base_vol = base.std()
pos_vol = pos.std()

# Tail risk metrics
tail_risk = {   
    'neg_mean': neg_mean,
    'not_neg_mean': not_neg_mean,
    'base_mean': base_mean,
    'pos_mean': pos_mean,
    'neg_vol': neg_vol,
    'not_neg_vol': not_neg_vol,
    'base_vol': base_vol,
    'pos_vol': pos_vol
}

# Print results
for key, value in tail_risk.items():   
    print(f"{key}: {value:.4f}")

# Compare bottom decile vs not bottom decile
neg_vs_not_neg = ttest_ind(neg, not_neg, equal_var=False)
# Compare bottom decile vs top decile
neg_vs_pos = ttest_ind(neg, pos, equal_var=False)

print("="*70)
print(f"Bottom decile vs not bottom decile: {neg_vs_not_neg}")
print(f"Bottom decile vs top decile: {neg_vs_pos}")


Probability of negative return: 0.5130
Probability of negative return (bottom decile): 0.4286
Probability of negative return (top decile): 0.4911
neg_mean: 0.0024
not_neg_mean: -0.0056
base_mean: -0.0047
pos_mean: -0.0096
neg_vol: 0.0337
not_neg_vol: 0.0478
base_vol: 0.0466
pos_vol: 0.0579
Bottom decile vs not bottom decile: TtestResult(statistic=2.2593935694698075, pvalue=0.025157663591170814, df=166.26638089538184)
Bottom decile vs top decile: TtestResult(statistic=1.857099787043345, pvalue=0.06507863106837192, df=165.08047944107156)


### Analysis
- Extreme negative sentiment does not behave like a downside-risk signal. 
- Bottom-decile (lowest 10%) is associated with modest next-day reversal behavior relative to the rest of the sample (other 90%)
- The direct comparison between bottom and top 10% is directionally consistent, but **not** statistically significant (p > 0.05)

### Interim Conclusion
The downside-risk hypothesis is **not** supported.  
- Bottom-decile sentiment is associated w/ **lower** next-day downside probability
    - Higher mean next-day returns and lower volatility relative to the rest of the sample.

### Follow-up question
Does bottom-decile sentiment retain any incremental relationship with next-day returns after considering forward price momentum?

## Regression

In [65]:
# Regression: forward_return ~ momentum + neg_extreme
#
# - Y = forward_return_1d (next trading day's return; merged from daily panel).
# - Momentum = prior trading day's return (lag-1 daily return), not forward_return_1d.
# - neg_extreme: run the "Conditional Risk Test" cell first, or we define it below.
#
# Important: X and y must use the SAME rows — never .dropna() on X and y separately.

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Prior-day return as momentum (from same daily return series as calculations)
_mom = returns_daily[["price_date", "ticker", "return_1d"]].copy()
_mom["momentum_1d"] = _mom.groupby("ticker")["return_1d"].shift(1)

reg_df = df_mag7_neg_decile.merge(
    _mom[["price_date", "ticker", "momentum_1d"]],
    on=["price_date", "ticker"],
    how="left",
)

if "neg_extreme" not in reg_df.columns:
    _low = reg_df["sentiment_score"].quantile(0.1)
    reg_df["neg_extreme"] = reg_df["sentiment_score"] <= _low

reg_df["neg_extreme"] = reg_df["neg_extreme"].astype(float)

feature_cols = ["momentum_1d", "neg_extreme"]
target = "forward_return_1d"

fit_df = reg_df.dropna(subset=feature_cols + [target]).copy()
X = fit_df[feature_cols]
y = fit_df[target]

model = LinearRegression().fit(X, y)
pred = model.predict(X)
r2 = r2_score(y, pred)

print(f"n = {len(fit_df):,}")
print(f"coef momentum_1d = {model.coef_[0]:.6f}, neg_extreme = {model.coef_[1]:.6f}")
print(f"intercept = {model.intercept_:.6f}")
print(f"R² (in-sample) = {r2:.6f}")


n = 1,106
coef momentum_1d = 0.088386, neg_extreme = -0.001129
intercept = 0.006205
R² (in-sample) = 0.005696


In [68]:
# Interaction model
reg_df["neg_extreme_x_momentum"] = reg_df["neg_extreme"] * reg_df["momentum_1d"]

feature_cols = ["momentum_1d", "neg_extreme", "neg_extreme_x_momentum"]
target = "forward_return_1d"

fit_df = reg_df.dropna(subset=feature_cols + [target]).copy()
X = fit_df[feature_cols]
y = fit_df[target]

model = LinearRegression().fit(X, y)
pred = model.predict(X)
r2 = r2_score(y, pred)

print(f"n = {len(fit_df):,}")
print(f"coef momentum_1d = {model.coef_[0]:.6f}, neg_extreme = {model.coef_[1]:.6f}, neg_extreme_x_momentum = {model.coef_[2]:.6f}")
print(f"intercept = {model.intercept_:.6f}")
print(f"R² (in-sample) = {r2:.6f}")


n = 1,106
coef momentum_1d = 0.124623, neg_extreme = -0.001751, neg_extreme_x_momentum = -0.314112
intercept = 0.006243
R² (in-sample) = 0.012956


## Findings

Under extreme negative sentiment:
- The relationship between momentum and next-day returns **reverses**
- Positive momentum is associated with lower (or negative) expected returns
- Negative momentum is associated with higher (or less negative) expected returns

*While the R<sup>2</sup> is low (0.012956), this is relatively normal for financial data*
### Summary
Negative sentiment breaks momentum and induces reversal

## Visualization?

In [1]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# plt.figure(figsize=(8, 6))

# # TODO: Add a regplot to compare neg_extreme vs not_neg_extreme

# plt.title("Momentum vs Forward Return (Regime Split)")
# plt.xlabel("Prior-Day Return (%)")
# plt.ylabel("Forward-Day Return (%)")




## Conclusion

Summarize findings and answer the hypothesis question.

## Effect Size

[Describe the magnitude of the observed effect]

## Direction

[Positive, negative, or no relationship? As expected or contrary?]

## Statistical Confidence (if applicable)

[p-values, confidence intervals, R-squared, etc.]

## Answer to Hypothesis (if applicable)

[Clear statement: supported, not supported, or inconclusive]

## Next Steps

- [Follow-up analysis needed]
- [Additional data required]
- [Related hypotheses to investigate]